In [4]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import HashingTF, IDF, Tokenizer

spark = SparkSession.builder.appName('tfidf').getOrCreate()

sentenceDataFrame = spark.createDataFrame([
    (0.0, 'Hi I love UIC love love love'), 
    (1.0, 'I wish I could stay in UIC forever'),
    (2.0, 'The UIC Data Science Programme is awesome')
], ['label', 'sentence'])

sentenceDataFrame.show(truncate=False)

+-----+-----------------------------------------+
|label|sentence                                 |
+-----+-----------------------------------------+
|0.0  |Hi I love UIC love love love             |
|1.0  |I wish I could stay in UIC forever       |
|2.0  |The UIC Data Science Programme is awesome|
+-----+-----------------------------------------+



In [5]:
tokenizer = Tokenizer(inputCol = 'sentence', outputCol = 'words')
word_tf = tokenizer.transform(sentenceDataFrame)
word_tf.show(truncate=False)

+-----+-----------------------------------------+-------------------------------------------------+
|label|sentence                                 |words                                            |
+-----+-----------------------------------------+-------------------------------------------------+
|0.0  |Hi I love UIC love love love             |[hi, i, love, uic, love, love, love]             |
|1.0  |I wish I could stay in UIC forever       |[i, wish, i, could, stay, in, uic, forever]      |
|2.0  |The UIC Data Science Programme is awesome|[the, uic, data, science, programme, is, awesome]|
+-----+-----------------------------------------+-------------------------------------------------+



In [6]:
hashing_tf = HashingTF(inputCol = 'words', outputCol = 'tf')
featurized_tf = hashing_tf.transform(word_tf)
featurized_tf.select("tf").show(truncate=False)

+----------------------------------------------------------------------------------------+
|tf                                                                                      |
+----------------------------------------------------------------------------------------+
|(262144,[19036,33808,186480,245417],[1.0,1.0,4.0,1.0])                                  |
|(262144,[19036,20719,58672,73249,205069,245417,250855],[2.0,1.0,1.0,1.0,1.0,1.0,1.0])   |
|(262144,[95889,106841,160735,190787,201539,219897,245417],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
+----------------------------------------------------------------------------------------+



In [7]:
idf = IDF(inputCol = 'tf', outputCol = 'tfidf')

In [8]:
idf_model = idf.fit(featurized_tf)
rescaled_tf = idf_model.transform(featurized_tf)
rescaled_tf.select('tf').show(truncate=False)

+----------------------------------------------------------------------------------------+
|tf                                                                                      |
+----------------------------------------------------------------------------------------+
|(262144,[19036,33808,186480,245417],[1.0,1.0,4.0,1.0])                                  |
|(262144,[19036,20719,58672,73249,205069,245417,250855],[2.0,1.0,1.0,1.0,1.0,1.0,1.0])   |
|(262144,[95889,106841,160735,190787,201539,219897,245417],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
+----------------------------------------------------------------------------------------+



In [9]:
#print(type(featurized_tf))
t = featurized_tf.select(['tf']).collect()
t2 = spark.createDataFrame(t, ['tf'])
#type(t2)
t2.show(truncate=False)

idf_model = idf.fit(t2)
rescaled_tf = idf_model.transform(t2)
rescaled_tf.select('tfidf').show(truncate=False)

+----------------------------------------------------------------------------------------+
|tf                                                                                      |
+----------------------------------------------------------------------------------------+
|(262144,[19036,33808,186480,245417],[1.0,1.0,4.0,1.0])                                  |
|(262144,[19036,20719,58672,73249,205069,245417,250855],[2.0,1.0,1.0,1.0,1.0,1.0,1.0])   |
|(262144,[95889,106841,160735,190787,201539,219897,245417],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
+----------------------------------------------------------------------------------------+

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|tfidf                                                                                                                                                                             |
